In [1]:
from backend.utils.gmsh_function import *
from backend.src.radiation_algorithm.radiation_algorithm import *
from backend.utils.file_path import *

In [2]:
name = "monopole"
save_mesh_folder = 'data/gmsh_files/'

# Define the path to save the file
path = setup_save_file_paths(name, save_mesh_folder)

In [3]:
L_plate = 2.0
W_plate = 2.0
Hight_monopole = 1.0
W_monopole = 0.02

depart_plate_x = -L_plate / 2
depart_plate_y = -W_plate / 2
depart_plate_z = 0

depart_monopole_x = -W_monopole / 2
depart_monopole_y = 0
depart_monopole_z = 0

In [4]:
light_speed = 3e8

frequency = 75e6
wavelength = light_speed / frequency
print(f"wavelength = {wavelength} Meter")

initial_mesh_size = wavelength / 20
print(f"initial_mesh_size = {initial_mesh_size} Meter")

feed_point = np.array([[0, 0, 0]])

wavelength = 4.0 Meter
initial_mesh_size = 0.2 Meter


In [5]:
gmsh.initialize()
gmsh.model.add("Momopole_Antenna")
setup_performance_config()

plate = gmsh.model.occ.addRectangle(-L_plate / 2, -W_plate / 2, 0, L_plate, W_plate)

# Create the monopole
M_P1 = gmsh.model.occ.addPoint(depart_monopole_x, depart_monopole_y, depart_monopole_z)
M_P2 = gmsh.model.occ.addPoint(depart_monopole_x + W_monopole, depart_monopole_y, depart_monopole_z)
M_P3 = gmsh.model.occ.addPoint(depart_monopole_x + W_monopole, depart_monopole_y, depart_monopole_z + Hight_monopole)
M_P4 = gmsh.model.occ.addPoint(depart_monopole_x, depart_monopole_y, depart_monopole_z + Hight_monopole)

M_L1 = gmsh.model.occ.addLine(M_P1, M_P2)
M_L2 = gmsh.model.occ.addLine(M_P2, M_P3)
M_L3 = gmsh.model.occ.addLine(M_P3, M_P4)
M_L4 = gmsh.model.occ.addLine(M_P4, M_P1)

Loop_monopole = gmsh.model.occ.addCurveLoop([M_L1, M_L2, M_L3, M_L4])
Surface_monopole = gmsh.model.occ.addPlaneSurface([Loop_monopole])

# Merge the two surfaces
monopole_antenna, _ = gmsh.model.occ.fuse([(2, plate)], [(2, Surface_monopole)])

gmsh.model.occ.synchronize()

generate_and_save_mesh(path.geo, path.msh, initial_mesh_size)

gmsh.fltk.run()

gmsh.finalize()

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Geometry file saved in data/gmsh_files/monopole.brep successfully
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
Mesh file saved in data/gmsh_files/monopole.msh successfully


In [6]:
extract_msh_to_mat(path.msh, path.mat)

In [ ]:
matrice_z, voltage, current, surface_current_density = radiation_algorithm(path.mat, frequency, feed_point, excitation_unit_vector='z', gap_width=3*initial_mesh_size/2)

IndexError: index 1 is out of bounds for axis 0 with size 1